In [ ]:
# Código da documentação do LangGraph, fica muita coisa implícita
from langgraph.graph import StateGraph, MessagesState, START, END

def mock_llm(state: MessagesState):
    return {"messages": [{"role": "ai", "content": "hello world"}]}

graph = StateGraph(MessagesState)
graph.add_node(mock_llm)
graph.add_edge(START, "mock_llm")
graph.add_edge("mock_llm", END)
graph = graph.compile()

graph.invoke({"messages": [{"role": "user", "content": "hi!"}]})

/Users/felipe/Documents/ds/lib/python3.13/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


{'messages': [HumanMessage(content='hi!', additional_kwargs={}, response_metadata={}, id='9d645613-adb5-4954-bc42-bb6b1f95915c'),
  AIMessage(content='hello world', additional_kwargs={}, response_metadata={}, id='5904916c-56cb-4c32-98cf-ffab544922d5', tool_calls=[], invalid_tool_calls=[])]}

In [ ]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages # -> gera uma estrutura de mensagens para o estado
from langchain_ollama import ChatOllama

# Estado
class Estado(TypedDict):
    messages: Annotated[list, add_messages]

# Conecta ao Ollama
llm = ChatOllama(model="llama3.2:1b")

# Função que usa o modelo
def responder(estado: Estado):
    resposta = llm.invoke(estado["messages"])
    return {"messages": [resposta]}

# Cria e compila o grafo
grafo = StateGraph(Estado)
grafo.add_node("responder", responder)
grafo.add_edge(START, "responder")
grafo.add_edge("responder", END)

app = grafo.compile()

# Executa
resultado = app.invoke({"messages": [("user", "Olá! Como você está?")]})
print(resultado["messages"][-1].content)

# Fluxo do modelo
# Usuário → "Olá!"
#     ↓
# Estado: {"messages": [("user", "Olá!")]}
#     ↓
# Nó "responder" → chama llm.invoke()
#     ↓
# GPT recebe: [("user", "Olá!")]
#     ↓
# GPT retorna: "Olá! Como posso ajudar?"
#     ↓
# Estado: {"messages": [("user", "Olá!"), ("assistant", "Olá! Como posso ajudar?")]}
#     ↓
# Usuário ← resposta

# Conceitos do LangGraph
# State ->	Estrutura de dados que mantém o estado da conversa
# Node ->	Função que processa e modifica o estado
# Edge ->	Conexão direta entre nós
# Conditional Edge ->	Escolhe o próximo nó baseado na lógica
# Stream ->	Executa e retorna eventos passo a passo

Olá! Estou bem, obrigado por perguntar. Eu estou aqui para ajudar e conversar com você. Como posso ajudá-lo hoje?
